# 5. GPT-2 inference with metrics

In this notebook, we will use the GPT-2 model to generate detoxified texts and rank them using the metrics we introduced in the previous notebooks.


In [1]:
# Dynamicly load evaluation metrics and the model
%run -i ../src/models/evaluation.py
%run -i ../src/models/detoxGPT2.py

In [2]:
import numpy as np
import pandas as pd
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# Instantiate model
detoxGPT = detoxGPT2()

# Instantiate metric classes
similarity = Similarity()
toxicity = STAToxic()

In [4]:
prompt = "What a fucking stupid thing to say!"

In [5]:
# Pack the suggestions into a dataframe
df = pd.DataFrame(
    detoxGPT.get_detoxed_suggestions(prompt, max_length=len(prompt), device=DEVICE), columns=["suggestion"]
)

# Add empty column for each metric
metrics = ["wo", "cs", "bleu"]
df[metrics] = pd.DataFrame([[0] * len(metrics)], index=df.index, dtype=float)

# Generate toxicity report for each suggestion
toxicity_report = toxicity.toxicity_report(df["suggestion"])

for index, row in df.iterrows():
    df.loc[index, "wo"] = similarity.get_wo_score(prompt, row["suggestion"])
    df.loc[index, "cs"] = similarity.get_cosine_score(prompt, row["suggestion"])
    df.loc[index, "bleu"] = similarity.get_bleu_score(prompt, row["suggestion"])

# Concat with toxicity report
df = pd.concat([df, toxicity_report], axis=1)
df

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


,suggestion,wo,cs,bleu,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,what a bad way of saying it!,0.166667,0.833712,0.298183,0.028699,0.000435,0.001174,0.000191,0.003110,0.000167
1,what a nonsense to think about.,0.300000,0.903489,0.325800,0.249878,0.000077,0.003130,0.000026,0.017738,0.002868
2,what a thing that is to say!,0.400000,0.908885,0.509277,0.013593,0.001437,0.005581,0.000121,0.006205,0.000841
3,What,0.142857,0.558483,0.000431,0.037838,0.001227,0.005018,0.000238,0.006115,0.000733
4,"Oh, what a thing!",0.222222,0.833020,0.199415,0.070872,0.002673,0.027271,0.000204,0.016358,0.010157
5,"oh, what a thing",0.375000,0.775773,0.187500,0.070872,0.002673,0.027271,0.000204,0.016358,0.010157
6,what a disgusting thing to say!,0.444444,0.973424,0.602127,0.632291,0.009197,0.082570,0.005091,0.195250,0.030325
7,"Hang it!"""" and what a wicked thing to say",0.454545,0.833929,0.494169,0.079283,0.002704,0.020046,0.048217,0.028292,0.001627


In [6]:
# Calculate the score
metric_weights = {"wo": 0.1, "cs": 0.5, "bleu": 0.4}

# Toxicity report should be as low as possible
# Similarity metrics should be as high as possible
df["detox_score"] = 1 - np.mean(df[["toxic"]], axis=1)
df["similarity"] = df[metrics].dot(pd.Series(metric_weights))

# Final score
df["score"] = df[["detox_score", "similarity"]].mean(axis=1)

# Sort by score
df = df.sort_values(by=["score"], ascending=False)
df

,suggestion,wo,cs,bleu,toxic,severe_toxic,obscene,threat,insult,identity_hate,detox_score,similarity,score
2,what a thing that is to say!,0.400000,0.908885,0.509277,0.013593,0.001437,0.005581,0.000121,0.006205,0.000841,0.986407,0.698153,0.842280
7,"Hang it!"""" and what a wicked thing to say",0.454545,0.833929,0.494169,0.079283,0.002704,0.020046,0.048217,0.028292,0.001627,0.920717,0.660087,0.790402
0,what a bad way of saying it!,0.166667,0.833712,0.298183,0.028699,0.000435,0.001174,0.000191,0.003110,0.000167,0.971301,0.552796,0.762048
4,"Oh, what a thing!",0.222222,0.833020,0.199415,0.070872,0.002673,0.027271,0.000204,0.016358,0.010157,0.929128,0.518498,0.723813
5,"oh, what a thing",0.375000,0.775773,0.187500,0.070872,0.002673,0.027271,0.000204,0.016358,0.010157,0.929128,0.500386,0.714757
1,what a nonsense to think about.,0.300000,0.903489,0.325800,0.249878,0.000077,0.003130,0.000026,0.017738,0.002868,0.750122,0.612065,0.681093
3,What,0.142857,0.558483,0.000431,0.037838,0.001227,0.005018,0.000238,0.006115,0.000733,0.962162,0.293700,0.627931
6,what a disgusting thing to say!,0.444444,0.973424,0.602127,0.632291,0.009197,0.082570,0.005091,0.195250,0.030325,0.367709,0.772008,0.569858


In [7]:
# Print the suggestion with the highest score
suggestion = df.iloc[0]["suggestion"]
print(f"{prompt} -> {suggestion}")

What a fucking stupid thing to say! -> what a thing that is to say!
